# 04 — Bar-Level vs Engine Path

Both backtest paths share the same signal, the same weighting rule (`1 / N_universe`
per in-signal symbol with uninvested capital sitting as cash), and the same fill
convention (open price on entry/exit bars).  This notebook runs them on identical
inputs and compares the resulting equity curves.

| | `BarBacktest` + `SignalAllocationPerformance` | `BacktestEngine` + `TrendSignalStrategy` |
|---|---|---|
| Output | Normalised equity (starts at 1) | Dollar NAV (starts at capital) |
| Fill | Open on entry/exit, prev-close on held | Open on entry/exit, close on held |
| Trade log | No | Yes |
| Tearsheet | `SignalAllocationPerformance.portfolio_metrics()` | `PerformanceCharts(result).tearsheet()` |

In [ ]:
import pandas as pd
import plotly.graph_objects as go

from hailmary.data.providers import YahooFinanceProvider
from hailmary.models import TrendSignal
from hailmary.backtest.signal_backtest import BarBacktest
from hailmary.backtest import BacktestEngine, TrendSignalStrategy
from hailmary.analytics.signal_analytics import SignalAllocationPerformance
from hailmary.viz.theme import PALETTE, apply_theme

In [ ]:
symbols = ["BTC-USD", "ETH-USD", "SOL-USD"]
start   = pd.Timestamp("2022-01-01")
end     = pd.Timestamp("2024-01-01")

signal = TrendSignal(ma_window=200)
yahoo  = YahooFinanceProvider()

fetch_start = start - pd.offsets.BDay(signal.warmup)
bars        = yahoo.get_bars(symbols, start=fetch_start, end=end)
signal_df   = signal.run(bars, trim_start=start)

print(f"Universe     : {symbols}")
print(f"Period       : {start.date()} → {end.date()}")
print(f"Trading days : {signal_df.index.get_level_values('timestamp').nunique()}")

## 1. Bar-Level Path

In [ ]:
bt_result = BarBacktest().run(signal_df)
alloc = SignalAllocationPerformance(bt_result)

bar_equity = alloc.portfolio_equity(method="mtc")
bar_equity.head(3)

## 2. Engine Path

In [ ]:
result = BacktestEngine(
    TrendSignalStrategy(signal_df),
    bars=bars,
    initial_capital=1_000_000,
    fill_mode="mtc",
).run(verbose=False)

# Normalise NAV to 1.0 for comparison
engine_equity = result.nav / result.nav.iloc[0]
engine_equity.head(3)

## 3. Overlay

Both curves should track closely.  Small divergences arise from a structural difference
in held-bar accounting: the bar-level path uses `prev-close → close` for held bars
(exact), while the engine computes held-bar returns implicitly from NAV snapshots at
close — the same result, but the engine also carries cash at the exact position size
rather than a notional weight, so rounding can introduce minor drift over long runs.

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=bar_equity.index, y=bar_equity,
    name="BarBacktest (MTC)",
    line={"color": PALETTE["accent_blue"], "width": 2},
))
fig.add_trace(go.Scatter(
    x=engine_equity.index, y=engine_equity,
    name="BacktestEngine (MTC)",
    line={"color": PALETTE["accent_green"], "width": 1.5, "dash": "dot"},
))

apply_theme(fig, "Bar-Level vs Engine Path — Normalised Equity (MTC)", height=460)
fig.update_layout(yaxis_title="Equity (normalised to 1.0)")
fig.show()

## 4. Performance Metrics Side by Side

In [ ]:
bar_metrics    = alloc.portfolio_metrics(method="mtc")
engine_metrics = result.summary()

comparison = pd.DataFrame({
    "BarBacktest": [
        f"{bar_metrics.total_return:.2%}",
        f"{bar_metrics.cagr:.2%}",
        f"{bar_metrics.annualised_vol:.2%}",
        f"{bar_metrics.sharpe:.2f}",
        f"{bar_metrics.max_drawdown:.2%}",
    ],
    "BacktestEngine": [
        engine_metrics["Total Return"],
        engine_metrics["Ann. Return"],
        engine_metrics["Ann. Volatility"],
        engine_metrics["Sharpe Ratio"],
        engine_metrics["Max Drawdown"],
    ],
}, index=["Total Return", "CAGR", "Ann. Volatility", "Sharpe", "Max Drawdown"])

comparison